# validate_stage4
CPU coordinator. Run cells 2, 3 and 4 in order. Configure only cell 3. Use the pinned runtime and exact Stage 4 integration commit.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage4')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 4 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Drazr/Research.git',str(REPO)],check=True)
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git','fetch','origin',CODE_REF],cwd=REPO,check=True)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=REPO,text=True).strip()
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-stage2-cpu.txt')],check=True)
print('Fixed code commit:',CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
STAGE4_PLAN = PHASE1_ROOT / 'REPLACE_STAGE4_PLAN/_COMPLETE.json'
INVENTORY = PHASE1_ROOT / 'REPLACE_STAGE4_WORKER_INVENTORY.json'


In [ ]:
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
for path in (STAGE4_PLAN, INVENTORY):
    if not path.is_file():
        raise RuntimeError(f'Set the explicit mounted artifact in cell 3: {path}')
command = [sys.executable, '-u', '-m', 'src.stage4.operations', 'validate',
           '--phase1-root', str(PHASE1_ROOT), '--storage-scope', 'persistent-drive']
subprocess.run(command + ['--manifest', str(STAGE4_PLAN), '--inventory', str(INVENTORY)], cwd=REPO, env=os.environ.copy(), check=True)
